In [1]:
import re
from collections import defaultdict, Counter

# 1. Define our training corpus (The text the model learns from)
corpus = """
Artificial intelligence and machine learning are transforming software development.
The software development life cycle is becoming more automated.
Python and C++ are popular programming languages for data science.
Deep learning models require massive amounts of data.
Machine learning algorithms predict future trends based on historical data.
The development of artificial intelligence requires careful data science practices.
"""

# 2. Text Preprocessing Function
def clean_and_tokenize(text):
    # Convert to lowercase and remove punctuation using regex
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    # Split into a list of words
    return text.split()

tokens = clean_and_tokenize(corpus)
print(f"Total tokens in corpus: {len(tokens)}")
print(f"First 10 tokens: {tokens[:10]}")

Total tokens in corpus: 56
First 10 tokens: ['artificial', 'intelligence', 'and', 'machine', 'learning', 'are', 'transforming', 'software', 'development', 'the']


In [2]:
# 3. Initialize dictionaries to hold our N-gram counts
# We use defaultdict(Counter) so we can easily count frequencies of 'next words'
unigrams = Counter(tokens)
bigrams = defaultdict(Counter)
trigrams = defaultdict(Counter)

# 4. Populate the N-gram dictionaries
for i in range(len(tokens) - 2):
    w1, w2, w3 = tokens[i], tokens[i+1], tokens[i+2]

    # Bigram: Given w1, count w2
    bigrams[w1][w2] += 1

    # Trigram: Given (w1, w2), count w3
    trigrams[(w1, w2)][w3] += 1

# Manually add the very last bigram (since the loop stops 2 words early)
bigrams[tokens[-2]][tokens[-1]] += 1

print("--- N-Gram Dictionaries Built ---")
print(f"Unique words (Unigrams): {len(unigrams)}")
print(f"Trigram example for ('machine', 'learning'): {trigrams[('machine', 'learning')]}")

--- N-Gram Dictionaries Built ---
Unique words (Unigrams): 40
Trigram example for ('machine', 'learning'): Counter({'are': 1, 'algorithms': 1})


In [3]:
def smart_autocomplete(user_input, top_k=3):
    input_tokens = clean_and_tokenize(user_input)

    if len(input_tokens) == 0:
        # If input is empty, return the most common words overall
        return [word for word, count in unigrams.most_common(top_k)]

    # Extract the last two words for our context
    w1 = input_tokens[-2] if len(input_tokens) >= 2 else None
    w2 = input_tokens[-1]

    predictions = []

    # 2. STEP 1: Try Trigram Model (Highest Accuracy)
    if w1 and (w1, w2) in trigrams:
        print("Model used: Trigram (Looked at last 2 words)")
        # Get the most common next words
        predictions = [word for word, count in trigrams[(w1, w2)].most_common(top_k)]
        return predictions

    # 3. STEP 2: Backoff to Bigram Model (Medium Accuracy)
    if w2 in bigrams:
        print("Model used: Bigram (Looked at last 1 word)")
        predictions = [word for word, count in bigrams[w2].most_common(top_k)]
        return predictions

    # 4. STEP 3: Backoff to Unigram Model (Fallback)
    # If the user typed a completely unknown word, suggest the most common overall words
    print("Model used: Unigram Fallback (Unknown context)")
    return [word for word, count in unigrams.most_common(top_k)]

In [4]:
print("--- Auto-Complete Tests ---\n")

test_1 = "artificial"
print(f"User types: '{test_1}'")
print(f"Suggestions: {smart_autocomplete(test_1)}\n")

test_2 = "machine learning"
print(f"User types: '{test_2}'")
print(f"Suggestions: {smart_autocomplete(test_2)}\n")

test_3 = "the software"
print(f"User types: '{test_3}'")
print(f"Suggestions: {smart_autocomplete(test_3)}\n")

test_4 = "completelyunknownword"
print(f"User types: '{test_4}'")
print(f"Suggestions: {smart_autocomplete(test_4)}\n")

--- Auto-Complete Tests ---

User types: 'artificial'
Model used: Bigram (Looked at last 1 word)
Suggestions: ['intelligence']

User types: 'machine learning'
Model used: Trigram (Looked at last 2 words)
Suggestions: ['are', 'algorithms']

User types: 'the software'
Model used: Trigram (Looked at last 2 words)
Suggestions: ['development']

User types: 'completelyunknownword'
Model used: Unigram Fallback (Unknown context)
Suggestions: ['data', 'learning', 'development']

